# RWB «Отгрузки без простоев» — решение, версия 2

Прогноз `target_1h` на 8 получасов вперёд для 1000 маршрутов от точки 01.11.2025 10:30. Метрика — **WAPE + |Relative Bias|**, меньше — лучше.

| | Score |
|---|---|
| Baseline организаторов (Ridge), на нашей валидации | 0.3674 |
| v1: XGBoost + Huber + MLP (`solution_v1.ipynb`) | валидация 0.3269 / 0.3291, **лидерборд 0.324** |
| **v2: + усреднение seed'ов + Tweedie (этот ноутбук)** | валидация 0.3268 / 0.3288, **лидерборд 0.323** |

Два числа валидации: первое — веса ансамбля подобраны на всех 14 днях (оптимистично), второе — перекрёстно: веса с одной недели проверены на другой.

**Что изменилось относительно v1.** Состав моделей тот же по сути, но XGBoost и MLP усреднены по трём seed'ам, и добавлен LightGBM с функцией потерь Tweedie. Tweedie предсказывает среднее, а не медиану, поэтому ему не нужна калибровка множителем, и на тесте его сумма совпала с суммой калиброванного ансамбля v1 — независимая проверка множителя. Прирост на лидерборде 0.001 — в пределах шума одного дня; главная ценность v2 в меньшем разбросе, а не в score.

Ноутбук — «тонкий»: весь код в `src/rwb/` и `scripts/`, здесь он вызывается и объясняется. Тяжёлые шаги (обучение восьми моделей, ~2.5 часа на RTX 4060 Laptop) берут сохранённые предсказания из `models/oof/`, если они есть; `RETRAIN = True` пересчитывает всё заново. Разделы 1–4 совпадают с `solution_v1.ipynb` и даны кратко.

1. Данные, метрика, получасовые отгрузки, валидация (кратко)
2. Признаки и цель обучения
3. Модели v2 и результаты по фолдам
4. Ансамбль: веса и калибровка, перекрёстно
5. Разбор ошибок
6. Итоговый прогноз и файл посылки
7. Что ещё проверено после v1 и не сработало

In [1]:
import sys, json, logging, time, itertools
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent  # запуск из notebooks/
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from rwb.data import HORIZON, load_panel, make_submission, wape_rbias
from rwb.features import DAY, build, cumsums
from rwb.backtest import default_folds, run, score_origins, truth_for
from rwb.models import GBMConfig, gbm_fit_predict, naive

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s", datefmt="%H:%M:%S")
RETRAIN = False          # True — переобучить модели, не используя сохранённые предсказания
OOF = ROOT / "models/oof"
OOF.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

## 1. Данные, метрика, получасы, валидация — кратко

- Train: 1000 маршрутов × 4630 получасов (28.07–01.11.2025 10:30), шесть статусов и `target_1h`. Держим как матрицы «маршрут × время».
- Метрика по всем 8000 точкам: $\frac{\sum|y-\hat y|}{\sum y} + \frac{|\sum \hat y - \sum y|}{\sum y}$ — ошибки точек плюс суммарное завышение/занижение.
- `target_1h[t] = x[t] + x[t-1]`: часовая сумма двух получасов при шаге в полчаса. Получасы `x` восстанавливаются знакопеременной суммой (константа на маршрут зажата условием $x \ge 0$), только по прошлому. На горизонте +30 минут половина ответа известна.
- Валидация: 10:30 в каждый из 14 предыдущих дней, два фолда по неделе; модель фолда обучается только на метках до первой точки фолда. `tests/test_features.py` портит будущее и проверяет, что признаки не меняются.

In [2]:
panel = load_panel()
cums = cumsums(panel)
folds = default_folds(panel)
for f in folds:
    days = [str(panel.times[o].date()) for o in f.val_origins]
    print(f"{f.name}: обучение на метках до {panel.times[f.label_cutoff]}, валидация в 10:30 на {days[0]} … {days[-1]}")
rows = []
for kind, descr in [("day", "тот же получас вчера"), ("slot7", "среднее того же получаса за 7 дней"), ("base", "сезонная база")]:
    sc = []
    for f in folds:
        sc += score_origins(naive(panel, f.val_origins, kind, cums), truth_for(panel, f.val_origins))
    rows.append((kind, descr, np.mean(sc)))
pd.DataFrame(rows, columns=["naive", "что это", "score"]).round(4)

fold0: обучение на метках до 2025-10-25 10:30:00, валидация в 10:30 на 2025-10-25 … 2025-10-31
fold1: обучение на метках до 2025-10-18 10:30:00, валидация в 10:30 на 2025-10-18 … 2025-10-24


,naive,что это,score
0,day,тот же получас вчера,0.4976
1,slot7,среднее того же получаса за 7 дней,0.3796
2,base,сезонная база,0.3638


## 2. Признаки и цель обучения

Для каждого горизонта `h` — таблица «маршрут × точка прогноза», всё только из данных до точки `o` (`rwb.features.build`): лаги и окна `y`, те же получасы в прошлые дни и недели, восстановленные получасы `x` (для `h=1` — известная половина ответа), статусы и их отношения, календарь.

**Цель** — поправка к сезонной базе в долях недельного уровня маршрута: $(y - \text{base})/\text{scale}$. Взвешенный `scale` L1 по ней равен числителю WAPE. В v2 добавлен режим `--target ratio`: неотрицательная цель $y/\text{scale}$ без базы, под Tweedie-распределение (`rwb.models.GBMConfig.target`).

In [3]:
X, names, base = build(panel, cums, np.array([folds[0].val_origins[0]]), h=1)
print(f"h=1: матрица {X.shape} (маршрутов × признаков)")
groups = {}
for n in names:
    groups.setdefault(n.split("_")[0], []).append(n)
pd.Series({k: len(v) for k, v in groups.items()}).sort_values(ascending=False).to_frame("признаков").T

h=1: матрица (1000, 119) (маршрутов × признаков)


,lag,x,rm,rs,sd,s5,s2,s1,s4,s3,...,dow,yday,lweek,slot14,d2,log,route,d1,slotw4,base
признаков,13,13,9,8,7,5,5,5,5,5,...,2,2,2,1,1,1,1,1,1,1


## 3. Модели v2

| Член ансамбля | Из чего | Что это |
|---|---|---|
| `xgb_lr_bag3` | `xgb_lr`, `xgb_lr_s1`, `xgb_lr_s2` | XGBoost (GPU), L1, по модели на горизонт: 900 деревьев, lr 0.02, 63 листа, 28 дней; три seed'а усреднены |
| `lgbm_huber` | `lgbm_huber` | LightGBM, Huber, сильная регуляризация (31 лист, ≥1000 объектов в листе) |
| `mlp_bag3` | `mlp_big`, `mlp_big_s1`, `mlp_big_s2` | MLP (GPU) с loss = WAPE по батчу, 84 дня истории; три seed'а усреднены |
| `lgbm_tweedie` | `lgbm_tweedie` | LightGBM, Tweedie (p = 1.5) на цели `ratio` — предсказывает среднее, калибровка не нужна |

Усреднение seed'ов делается до подбора весов, иначе оптимизатор делит вес между почти одинаковыми моделями произвольно (`scripts/bag_oof.py`). Конфигурации ниже — те же, что в README.

In [4]:
XGB = dict(lib="xgb", days=28, stride=2, rounds=900, lr=0.02, leaves=63, min_child=200)
MLP = dict(days=84, recent=192, hidden=1024, dropout=0.2, epochs=12)
RUNS = {  # имя прогона -> конфигурация (seed задаётся отдельно)
    "xgb_lr": (GBMConfig(**XGB, seed=0), None), "xgb_lr_s1": (GBMConfig(**XGB, seed=1), None), "xgb_lr_s2": (GBMConfig(**XGB, seed=2), None),
    "lgbm_huber": (GBMConfig(lib="lgbm", days=28, stride=4, rounds=400, lr=0.05, leaves=31, min_child=1000,
                             objective="huber", extra={"num_threads": 8, "alpha": 1.0}), None),
    "lgbm_tweedie": (GBMConfig(lib="lgbm", days=28, stride=4, rounds=600, lr=0.05, leaves=31, min_child=1000,
                               objective="tweedie", target="ratio", extra={"num_threads": 8, "tweedie_variance_power": 1.5}), None),
    "mlp_big": (None, 0), "mlp_big_s1": (None, 1), "mlp_big_s2": (None, 2),
}
MEMBERS = {"xgb_lr_bag3": ["xgb_lr", "xgb_lr_s1", "xgb_lr_s2"], "lgbm_huber": ["lgbm_huber"],
           "mlp_bag3": ["mlp_big", "mlp_big_s1", "mlp_big_s2"], "lgbm_tweedie": ["lgbm_tweedie"]}

def fit_predict(run, label_cutoff, origins):
    cfg, seed = RUNS[run]
    if cfg is not None:
        return gbm_fit_predict(panel, cums, label_cutoff, origins, cfg)
    from rwb.nn import NNConfig, nn_fit_predict
    return nn_fit_predict(panel, cums, label_cutoff, origins, NNConfig(**MLP, seed=seed))

def backtest(name):
    path = OOF / f"{name}.npz"
    if path.exists() and not RETRAIN:
        z = np.load(path)
        return {f.name: z[f"pred_{f.name}"] for f in folds}, {f.name: z[f"truth_{f.name}"] for f in folds}
    t0 = time.time()
    res = run(panel, folds, lambda f: fit_predict(name, f.label_cutoff, f.val_origins))
    np.savez_compressed(path, **{f"pred_{k}": v for k, v in res.preds.items()}, **{f"truth_{k}": v for k, v in res.truth.items()})
    print(f"{name}: {time.time() - t0:.0f} c")
    return res.preds, res.truth

run_preds, truth = {}, None
for r in RUNS:
    run_preds[r], truth = backtest(r)
preds = {m: {f: np.mean([run_preds[r][f] for r in rs], axis=0) for f in truth} for m, rs in MEMBERS.items()}

rows = []
for name in list(RUNS) + list(MEMBERS):
    src = run_preds.get(name) or preds[name]
    raw = {f: np.mean(score_origins(src[f], truth[f])) for f in truth}
    rows.append({"модель": name, **{f"{f} (без калибровки)": v for f, v in raw.items()}, "mean": np.mean(list(raw.values()))})
pd.DataFrame(rows).set_index("модель").round(4)

,fold0 (без калибровки),fold1 (без калибровки),mean
модель,,,
xgb_lr,0.3346,0.3433,0.3390
xgb_lr_s1,0.3348,0.3430,0.3389
xgb_lr_s2,0.3349,0.3430,0.3390
lgbm_huber,0.3291,0.3276,0.3284
lgbm_tweedie,0.3302,0.3254,0.3278
mlp_big,0.3392,0.3454,0.3423
mlp_big_s1,0.3398,0.3447,0.3422
mlp_big_s2,0.3389,0.3433,0.3411
xgb_lr_bag3,0.3347,0.3431,0.3389


Без калибровки XGBoost и MLP хуже Tweedie и Huber: L1 даёт медиану, а распределение отгрузок скошено вправо, так что сумма занижена примерно на 2% и |Relative Bias| штрафует. Это чинит общий множитель на следующем шаге. Tweedie и Huber ближе к среднему и множителя почти не требуют. Усреднение seed'ов снижает ошибку модели лишь на ~0.0002: модели сильно регуляризованы и данных много, разброс между seed'ами мал.

## 4. Ансамбль

Доли четырёх членов (на симплексе, через softmax) и общий множитель подбираются минимизацией метрики (`scripts/ensemble.py`). Оценка перекрёстная: веса с одной недели, метрика на другой. Итоговые веса — на всех 14 днях. Это 5 чисел по 14 × 8000 точкам; тестовые метки не используются нигде.

In [5]:
from ensemble import fit_weights     # scripts/ensemble.py

NAMES = list(MEMBERS)
fold_names = sorted(truth)
P = {f: np.stack([preds[n][f] for n in NAMES], axis=-1) for f in fold_names}
cross = {}
for fit_f, eval_f in itertools.permutations(fold_names, 2):
    w, _ = fit_weights(P[fit_f], truth[fit_f], per_h_scale=False)
    cross[f"веса с {fit_f} → оценка на {eval_f}"] = (np.mean(score_origins(P[eval_f] @ w, truth[eval_f])), w)
for k, (s, w) in cross.items():
    print(f"{k}: {s:.4f}   веса {np.round(w, 3)}")
P_all = np.concatenate([P[f] for f in fold_names]); Y_all = np.concatenate([truth[f] for f in fold_names])
W, _ = fit_weights(P_all, Y_all, per_h_scale=False)
print(f"\nперекрёстно: {np.mean([v[0] for v in cross.values()]):.4f}")
print(f"веса на всех 14 днях: {dict(zip(NAMES, np.round(W, 3)))}, сумма = множитель калибровки = {W.sum():.3f}")
print(f"на тех же 14 днях (оптимистично): {np.mean(score_origins(P_all @ W, Y_all)):.4f}")

веса с fold0 → оценка на fold1: 0.3281   веса [0.899 0.    0.118 0.   ]
веса с fold1 → оценка на fold0: 0.3296   веса [0.548 0.105 0.254 0.115]



перекрёстно: 0.3288
веса на всех 14 днях: {'xgb_lr_bag3': np.float64(0.673), 'lgbm_huber': np.float64(0.143), 'mlp_bag3': np.float64(0.107), 'lgbm_tweedie': np.float64(0.097)}, сумма = множитель калибровки = 1.020
на тех же 14 днях (оптимистично): 0.3268


## 5. Разбор ошибок

WAPE по дням стабилен (0.30–0.32), а смещение скачет от −0.046 до +0.040: суммарный объём дня не предсказуем заранее — ни из истории маршрутов, ни отдельной моделью суммарного объёма сети (проверено). Это и есть потолок для всех моделей.

In [6]:
origins = np.concatenate([f.val_origins for f in folds])
blend = P_all @ W
rows = []
for i, o in enumerate(origins):
    p, y = blend[i], Y_all[i]
    wape = np.abs(p - y).sum() / y.sum(); rb = (p.sum() - y.sum()) / y.sum()
    rows.append({"день": panel.times[o].strftime("%Y-%m-%d %a"), "WAPE": wape, "RelBias": rb, "score": wape + abs(rb)})
df = pd.DataFrame(rows).set_index("день")
print(f"средний WAPE {df.WAPE.mean():.4f}  средний |bias| {df.RelBias.abs().mean():.4f}  score {df.score.mean():.4f}")
print("WAPE по горизонтам:", np.round([np.mean([np.abs(blend[i][:, h] - Y_all[i][:, h]).sum() / Y_all[i][:, h].sum()
                                                 for i in range(len(origins))]) for h in range(HORIZON)], 3))
df.round(4)

средний WAPE 0.3146  средний |bias| 0.0122  score 0.3268
WAPE по горизонтам: [0.23  0.3   0.318 0.32  0.338 0.346 0.336 0.334]


,WAPE,RelBias,score
день,,,
2025-10-25 Sat,0.3202,0.0079,0.3281
2025-10-26 Sun,0.3232,-0.0021,0.3253
2025-10-27 Mon,0.3200,-0.0007,0.3207
2025-10-28 Tue,0.3144,-0.0047,0.3191
2025-10-29 Wed,0.3219,0.0397,0.3615
2025-10-30 Thu,0.3043,0.0120,0.3163
2025-10-31 Fri,0.3172,0.0148,0.3320
2025-10-18 Sat,0.3160,-0.0026,0.3186
2025-10-19 Sun,0.3108,-0.0203,0.3311


## 6. Итоговый прогноз

Каждый прогон обучается на всех данных (метки до 01.11.2025 10:30), seed'ы усредняются, члены смешиваются с весами `W`. Сохранённые тестовые прогнозы — `models/oof/<прогон>_test.npy`.

In [7]:
last = panel.n_times - 1
test_runs = {}
for r in RUNS:
    path = OOF / f"{r}_test.npy"
    if path.exists() and not RETRAIN:
        test_runs[r] = np.load(path)
    else:
        test_runs[r] = fit_predict(r, last, np.array([last]))[0]
        np.save(path, test_runs[r])
test_members = np.stack([np.mean([test_runs[r] for r in MEMBERS[m]], axis=0) for m in NAMES], axis=-1)
final = test_members @ W          # (R, 8)

sub = make_submission(panel, final)
out = ROOT / "submissions/solution_v2_notebook.csv"
sub.to_csv(out, index=False)
assert len(sub) == 8000 and set(sub["id"]) == set(panel.test["id"]) and (sub["y_pred"] >= 0).all() and np.isfinite(sub["y_pred"]).all()
print(f"{out.name}: {len(sub)} строк, сумма прогноза {sub.y_pred.sum():.3e}")
tw = test_runs["lgbm_tweedie"].sum()
print(f"сумма Tweedie без калибровки: {tw:.3e} — независимая проверка множителя {W.sum():.3f}")
for ref_name, lb in [("ensemble_v2.csv", 0.323), ("ensemble_v1.csv", 0.324)]:
    ref = ROOT / "submissions" / ref_name
    if ref.exists():
        v = pd.read_csv(ref).set_index("id").loc[sub["id"], "y_pred"].to_numpy()
        print(f"отличие от {ref_name} (лидерборд {lb}): средн. |Δ|/y = {np.abs(v - sub.y_pred.to_numpy()).sum() / v.sum():.2e}")
sub.head()

solution_v2_notebook.csv: 8000 строк, сумма прогноза 2.222e+09
сумма Tweedie без калибровки: 2.221e+09 — независимая проверка множителя 1.020
отличие от ensemble_v2.csv (лидерборд 0.323): средн. |Δ|/y = 0.00e+00
отличие от ensemble_v1.csv (лидерборд 0.324): средн. |Δ|/y = 4.52e-03


,id,y_pred
0,0,106312.241962
1,1,138608.595035
2,2,172299.136125
3,3,165591.752495
4,4,153883.090806


## 7. Что проверено после v1 и не сработало

Всё — на той же валидации, в сравнении с 0.3269 / 0.3291 у v1.

| Идея | Score | Итог |
|---|---|---|
| Квантильная функция потерь (α = 0.53, 0.56) вместо L1 + множитель | 0.331 | хуже |
| Модель получасов `x` на всех горизонтах со склейкой в часы (`--target x`) | 0.3285 | хуже; на первом горизонте (`--target x1`) WAPE 0.228 вместо 0.230, на сумму ~0.0001 |
| Ridge на тех же 119 признаках (`--lib ridge`) | 0.3310 | вес в ансамбле 0.01 |
| LightGBM random forest | 0.3362 | хуже |
| CatBoost GPU с MAE | — | забирает всю видеопамять, снят |
| GRU по окну истории (`--arch gru`) | 0.3298 | лучше MLP как одиночная модель, в ансамбле прироста нет |
| Отдельная модель суммарного объёма сети для снятия дневного смещения | хуже суммы маршрутов | смещение не предсказуемо |
| Сглаживание по горизонтам, веса на каждый горизонт, множители и сдвиги по маршрутам | хуже перекрёстно | — |

Ошибки всех классов моделей (деревья, MLP, GRU, Chronos-2) коррелируют на ~0.95: остаток — шум получасовых отгрузок. Лидерборд — один день с разбросом дневного смещения ±0.01–0.05, поэтому разницы порядка 0.001 между v1 и v2 на нём не значимы.